In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
import requests as req

C:\Users\amine\AppData\Local\Temp\ipykernel_20496\1227859210.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import os
from dotenv import load_dotenv

def get_api_key():

    load_dotenv()
    api_key = os.getenv("YOUTUBE_API_KEY")
    if not api_key:
        raise ValueError("API Key not found. Please check your .env file.")
    return api_key


In [3]:
load_dotenv()
base_url = os.getenv("BASE_URL")
channel_handle = os.getenv("CHANNEL_HANDLE")

### Channel chosen for the project is Kurzgesagt

In [4]:
def get_channel_id(handle):
    params = {
    "part": "id",
    "forHandle": handle
    #'forHandle': '@Kurzgesagt'
    }
    res = req.get(f"{base_url}/channels",params=params | {"key": get_api_key()}).json()

    return res['items'][0]['id']

channel_id = get_channel_id(channel_handle)
channel_id

'UCsXVk37bltHxD1rDPwtNM8Q'

In [5]:
def get_playlist_id(handle):
    params = {
    "part": "contentDetails",
    "forHandle": handle
    #'forHandle': '@Kurzgesagt'
    }
    res = req.get(f"{base_url}/channels",params=params | {"key": get_api_key()}).json()

    return res['items'][0]['contentDetails']['relatedPlaylists']['uploads']

playlist_id = get_playlist_id(channel_handle)
playlist_id

'UUsXVk37bltHxD1rDPwtNM8Q'

In [6]:
def get_playlist_page(playlistId, max_results=50, page_number=1):
    """
    Recupere les Ids des video dans une seul page specifiée.
    """
    if page_number < 1:
        raise ValueError("Page number must be 1 or greater.")

    current_page = 1
    next_page_token = None

    while current_page <= page_number:
        params = {
            "part": "contentDetails",
            "playlistId": playlistId,
            "maxResults": max_results,
            "pageToken": next_page_token
        }

        response = req.get(f"{base_url}/playlistItems", params=params | {"key": get_api_key()})

        response.raise_for_status()
        data = response.json()

        if current_page == page_number:
            page_data = data.get('items', [])
            return [item['contentDetails']['videoId'] for item in page_data]

        next_page_token = data.get('nextPageToken')

        # If the API stops returning tokens before we reach our target page,
        # it means the user asked for a page that doesn't exist.
        if not next_page_token:
            print(f"Warning: Page {page_number} does not exist. The playlist only has {current_page} pages.")
            return []

        current_page += 1

# --- How to use it ---
# Get page 3 of Kurzgesagt's uploads, with 20 items per page
page_3_data = get_playlist_page(playlist_id, max_results=20, page_number=3)
page_3_data

['nDKHvoo4f-8',
 'Brm71uCWr-I',
 'pt0iG95Fsmc',
 'Ghy3qRR6kiw',
 'cE0zgN6pYOc',
 'ty0WVFjOkok',
 '_zfN9wnPvU0',
 'mxG0ts7qmq0',
 'VMm-U2pHrXE',
 'V7wlYQ2_Q8o',
 'Zpq1GbPqhy4',
 'ByD__-ECSeI',
 'dKm7T13X7n4',
 'XFODuRPv8Yw',
 'XunabxHzMuM',
 'K1J9LNElrso',
 'gtDKKJq9u30',
 'LbARorLmhsc',
 'UNunBadMlHQ',
 'u4LUix-BU0s']

In [7]:
def get_all_video_ids(playlistId):
    video_ids = []
    next_page_token = None

    while True:
        params = {
            "part": "contentDetails",
            "playlistId": playlistId,
            "maxResults": 50,
            "pageToken": next_page_token
        }

        # Appel API
        response = req.get(f"{base_url}/playlistItems", params=params | {"key": get_api_key()})
        data = response.json()

        # Extraction des IDs dans la page actuelle
        items = data.get('items', [])
        for item in items:
            video_ids.append(item['contentDetails']['videoId'])

        # On vérifie s'il y a une page suivante
        next_page_token = data.get('nextPageToken')

        # Si pas de token, on a fini !
        if not next_page_token:
            break

    return video_ids

all_videos_ids = get_all_video_ids(get_playlist_id(channel_handle))
all_videos_ids

['RniYUmKs93U',
 'z7Qp9Q_Br88',
 'FZbMPtLRr00',
 'dl0-TveDDGA',
 'VYhUa3WC4nU',
 'lRvN5SJR2LQ',
 'onFj_RD75nA',
 'h3DCdWyb0cc',
 'MX_r0KsMpho',
 'NXhKuMQRAZ4',
 'yDAAlojz8NU',
 'iZ9ImJQy9V4',
 '5Xd38M9zTjM',
 'YHlqwkdVda8',
 'MP4RUzRnobg',
 'KTu2V65Iiho',
 'qPSxUkssPmc',
 '5EPPBDdhe6o',
 'dwZgy7SNMRc',
 'mWN2IefeJow',
 'bRkxPsuVEeU',
 'zg40FHOAqC0',
 'D0WQK7cMJn8',
 'OU36JDGzXb0',
 'hDU9WcmQA2E',
 'sravy3Vjdlw',
 'pHJIhxZEoxg',
 'UnrtLi7YIY4',
 'UC5Mpc-GQCg',
 'H4KMQuyxH_Y',
 '1bQsvHqSfpU',
 'KNwMiydCYA4',
 'T6tCY6SOSXo',
 'zozEm4f_dlw',
 'omNvSHfIv7s',
 'ZSch_NgZpQs',
 'fAkMppZZiSI',
 'sOsqXKr4l30',
 'OxOx6--oN8I',
 'Mo1A45ShcMo',
 'nDKHvoo4f-8',
 'Brm71uCWr-I',
 'pt0iG95Fsmc',
 'Ghy3qRR6kiw',
 'cE0zgN6pYOc',
 'ty0WVFjOkok',
 '_zfN9wnPvU0',
 'mxG0ts7qmq0',
 'VMm-U2pHrXE',
 'V7wlYQ2_Q8o',
 'Zpq1GbPqhy4',
 'ByD__-ECSeI',
 'dKm7T13X7n4',
 'XFODuRPv8Yw',
 'XunabxHzMuM',
 'K1J9LNElrso',
 'gtDKKJq9u30',
 'LbARorLmhsc',
 'UNunBadMlHQ',
 'u4LUix-BU0s',
 'aOwmt39L2IQ',
 'x7IrzHMxEMM',
 '3D9uIn

### Video statistics

In [8]:
def get_video_stats(videoId):
    params = {
    "part": "statistics",
    "id": videoId
    }
    res = req.get(f"{base_url}/videos",params=params | {"key": get_api_key()}).json()

    return res['items'][0]['statistics']

video_stats = get_video_stats("XFODuRPv8Yw")
video_stats

{'viewCount': '714662',
 'likeCount': '43739',
 'favoriteCount': '0',
 'commentCount': '1317'}

### Video Topic

In [9]:
def get_video_topic(videoId):
    params = {
    "part": "topicDetails",
    "id": videoId
    }
    res = req.get(f"{base_url}/videos",params=params | {"key": get_api_key()}).json()

    return res["items"][0]["topicDetails"]

video_topic = get_video_topic("XFODuRPv8Yw")
video_topic

{'topicCategories': ['https://en.wikipedia.org/wiki/Casual_game',
  'https://en.wikipedia.org/wiki/Strategy_video_game',
  'https://en.wikipedia.org/wiki/Video_game_culture']}

In [10]:
def get_video_title(videoId):
    params = {
    "part": "snippet",
    "id": videoId
    }
    res = req.get(f"{base_url}/videos",params=params | {"key": get_api_key()}).json()

    return res["items"][0]["snippet"]['title']

video_title = get_video_title("XFODuRPv8Yw")
video_title

'Our PC Game Is Finally Here!'

In [11]:
def get_video_duration(videoId):
    params = {
    "part":"contentDetails",
    "id": videoId
    }
    res = req.get(f"{base_url}/videos",params=params | {"key": get_api_key()}).json()

    return res["items"][0]["contentDetails"]["duration"]

video_duration = get_video_duration("XFODuRPv8Yw")
video_duration

'PT1M8S'

In [12]:
def get_videos_data_batch(video_ids_list):

    ids_string = ",".join(video_ids_list)

    params = {
        "part": "snippet,contentDetails,statistics,topicDetails",
        "id": ids_string
    }

    res = req.get(f"{base_url}/videos", params=params | {"key": get_api_key()}).json()

    batch_data = []
    categorie_ids = set()
    # On boucle sur les vidéos renvoyées dans ce batch
    for item in res.get("items", []):
        snippet = item.get("snippet", {})
        contentDetails = item.get("contentDetails", {})
        statistics = item.get("statistics", {})
        topicDetails = item.get("topicDetails", {})

        video_data = {
            "videoId": item.get("id"),
            "title": snippet.get("title"),
            "publishedAt": snippet.get("publishedAt"),
            "duration": contentDetails.get("duration"),
            "viewCount": statistics.get("viewCount"),
            "likeCount": statistics.get("likeCount"),
            "dislikeCount": statistics.get("dislikeCount"),
            "commentCount": statistics.get("commentCount"),
            "thumbnailUrl": snippet.get("thumbnails", {}).get("medium", {}).get("url"),
            "topicCategories": topicDetails.get("topicCategories", [])
        }
        batch_data.append(video_data)

    return batch_data

In [13]:
def get_all_videos_dataset(all_videos_ids):
    final_dataset = []

    for i in range(0, len(all_videos_ids), 50):

        batch_ids = all_videos_ids[i:i+50]

        batch_data= get_videos_data_batch(batch_ids)

        final_dataset.extend(batch_data)

    return final_dataset

In [14]:
final = get_all_videos_dataset(all_videos_ids)

In [15]:
len(final)

358

In [16]:
final

[{'videoId': 'RniYUmKs93U',
  'title': 'Pigeons Are GPS Drones',
  'publishedAt': '2026-04-30T14:00:48Z',
  'duration': 'PT1M7S',
  'viewCount': '248242',
  'likeCount': '15874',
  'dislikeCount': None,
  'commentCount': '359',
  'thumbnailUrl': 'https://i.ytimg.com/vi/RniYUmKs93U/mqdefault.jpg',
  'topicCategories': []},
 {'videoId': 'z7Qp9Q_Br88',
  'title': 'Laughing Gas Is Not So Funny',
  'publishedAt': '2026-04-27T14:01:38Z',
  'duration': 'PT1M15S',
  'viewCount': '540231',
  'likeCount': '29784',
  'dislikeCount': None,
  'commentCount': '793',
  'thumbnailUrl': 'https://i.ytimg.com/vi/z7Qp9Q_Br88/mqdefault.jpg',
  'topicCategories': ['https://en.wikipedia.org/wiki/Entertainment']},
 {'videoId': 'FZbMPtLRr00',
  'title': 'Fast-Forward Your Life',
  'publishedAt': '2026-04-23T14:00:23Z',
  'duration': 'PT1M5S',
  'viewCount': '554287',
  'likeCount': '21352',
  'dislikeCount': None,
  'commentCount': '855',
  'thumbnailUrl': 'https://i.ytimg.com/vi/FZbMPtLRr00/mqdefault.jpg',
  

In [39]:
import json

def save_data_to_json(data, filepath):
    # 1. Extraire le chemin du dossier depuis le filepath complet
    directory = os.path.dirname(filepath)

    if directory:
        os.makedirs(directory, exist_ok=True)

    with open(filepath, 'w', encoding='utf-8') as file:

        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f"✅ Données sauvegardées avec succès dans : {filepath}")

In [40]:
final = get_all_videos_dataset(all_videos_ids)

save_data_to_json(final, "../data/raw/videos.json")

✅ Données sauvegardées avec succès dans : ../data/raw/videos.json
